In [6]:

!pip install -q pdfplumber sentence-transformers faiss-cpu transformers accelerate bitsandbytes tqdm arxiv gradio nltk scikit-learn


In [2]:

import os
import re
import hashlib
import pickle
import requests
from typing import List, Dict, Tuple
from tqdm.notebook import tqdm
import numpy as np
import pdfplumber
import arxiv
import torch
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from google.colab import files, drive

# Configuration
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # 384-dim, fast & good for prototyping
CHUNK_SIZE = 512                      # tokens
CHUNK_OVERLAP = 50
TOP_K = 5
DATA_DIR = "/content/rag_pdfs"
INDEX_PATH = "/content/faiss_index"
METADATA_PATH = "/content/chunks_metadata.pkl"

os.makedirs(DATA_DIR, exist_ok=True)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:

# Option A: Upload own PDFs via Colab file upload
print("Option 1: Upload your own PDF files")
uploaded = files.upload()   # Upload multiple PDFs
for fn in uploaded.keys():
    !mv "$fn" "{DATA_DIR}/"

# Option B: Download 20 research papers from arXiv (uncomment to use)

# print("Option 2: Downloading sample papers from arXiv...")
# query = "machine learning OR large language model"
# search = arxiv.Search(query=query, max_results=20, sort_by=arxiv.SortCriterion.SubmittedDate)
# for paper in tqdm(search.results(), total=20):
#     paper.download_pdf(dirpath=DATA_DIR, filename=f"{paper.get_short_id()}.pdf")

# List downloaded PDFs
pdf_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.pdf')]
print(f"Total PDFs to process: {len(pdf_files)}")

Option 1: Upload your own PDF files


Saving Blockchain security.pdf to Blockchain security.pdf
Total PDFs to process: 1


In [4]:

# PARSING & CLEANING (with deduplication)

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from a single PDF using pdfplumber."""
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Error parsing {pdf_path}: {e}")
    return text

def clean_text(text: str) -> str:
    """Remove boilerplate, extra spaces, and normalize."""
    # Remove headers/footers (simple heuristic: short lines with page numbers)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        # Skip page numbers, standalone dashes, or very short lines (<10 chars)
        if len(line) < 10 or re.match(r'^\s*\d+\s*$', line):
            continue
        cleaned_lines.append(line)
    text = ' '.join(cleaned_lines)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text

def get_doc_hash(text: str) -> str:
    """Generate SHA-256 hash of the document text for deduplication."""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

# Process all PDFs: extract, clean, deduplicate
documents = []      # list of (hash, text, source_file)
seen_hashes = set()

for pdf_file in tqdm(pdf_files, desc="Parsing PDFs"):
    path = os.path.join(DATA_DIR, pdf_file)
    raw_text = extract_text_from_pdf(path)
    if not raw_text:
        continue
    clean = clean_text(raw_text)
    doc_hash = get_doc_hash(clean)
    if doc_hash not in seen_hashes:
        seen_hashes.add(doc_hash)
        documents.append((doc_hash, clean, pdf_file))
    else:
        print(f"Skipping duplicate: {pdf_file}")

print(f"Unique documents after cleaning & dedup: {len(documents)}")

Parsing PDFs:   0%|          | 0/1 [00:00<?, ?it/s]

Unique documents after cleaning & dedup: 1


In [8]:

# SEMANTIC CHUNKING (with robust tokenization)

import re
import hashlib
import numpy as np
from typing import List, Tuple
from tqdm.notebook import tqdm

# ROBUST SENTENCE TOKENIZER (No NLTK required)

def simple_sentence_tokenize(text: str) -> List[str]:
    """
    Simple but effective sentence tokenizer using regex.
    Handles common abbreviations and edge cases.
    """
    # Common abbreviations that shouldn't end sentences
    abbreviations = r'\b(?:Mr|Mrs|Ms|Dr|Prof|Rev|Hon|Gen|Col|Maj|Capt|Sgt|Lt|Jr|Sr|vs|etc|e\.g|i\.e|al\.\.)\.[\.]?'

    # Remove the abbreviations temporarily to avoid false splits
    placeholder = "___ABBR___"
    abbr_matches = re.findall(abbreviations, text)
    for i, abbr in enumerate(abbr_matches):
        text = text.replace(abbr, f"{placeholder}_{i}")

    # Split on sentence endings (. ! ?) followed by space and capital letter or end of string
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"\'\(])', text)

    # Restore abbreviations
    for i, abbr in enumerate(abbr_matches):
        restored_text = []
        for sentence in sentences:
            sentence = sentence.replace(f"{placeholder}_{i}", abbr)
            restored_text.append(sentence)
        sentences = restored_text

    # Clean up empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences


class AdvancedTextSplitter:
    """
    Custom text splitter with multiple strategies:
    1. Sliding window with overlap
    2. Sentence-aware splitting (using custom tokenizer)
    3. Semantic boundary detection (optional)
    """

    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 50, respect_sentences: bool = True):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.respect_sentences = respect_sentences

    def split_text(self, text: str) -> List[str]:
        """Split text into chunks using the selected strategy."""
        if not text or not text.strip():
            return []

        if not self.respect_sentences:
            # Simple character-based sliding window
            return self._sliding_window_split(text)
        else:
            # Sentence-aware splitting
            return self._sentence_aware_split(text)

    def _sliding_window_split(self, text: str) -> List[str]:
        """Basic sliding window with overlap."""
        chunks = []
        start = 0
        text_len = len(text)

        while start < text_len:
            end = min(start + self.chunk_size, text_len)
            chunks.append(text[start:end])
            start += (self.chunk_size - self.chunk_overlap)

            if start >= text_len:
                break

        return chunks

    def _sentence_aware_split(self, text: str) -> List[str]:
        """Split at sentence boundaries while respecting chunk size."""
        # Use our custom sentence tokenizer (no NLTK needed)
        sentences = simple_sentence_tokenize(text)

        if not sentences:
            return []

        chunks = []
        current_chunk = []
        current_length = 0

        for sentence in sentences:
            sentence_len = len(sentence)

            # If a single sentence exceeds chunk_size, force split it
            if sentence_len > self.chunk_size:
                # Save current chunk if not empty
                if current_chunk:
                    chunks.append(' '.join(current_chunk))
                    # Keep overlap: last few sentences from previous chunk
                    overlap_sentences = []
                    overlap_len = 0
                    for s in reversed(current_chunk):
                        if overlap_len + len(s) <= self.chunk_overlap:
                            overlap_sentences.insert(0, s)
                            overlap_len += len(s)
                        else:
                            break
                    current_chunk = overlap_sentences
                    current_length = overlap_len

                # Split the long sentence into smaller pieces
                sub_chunks = self._split_long_sentence(sentence)
                for sub_chunk in sub_chunks[:-1]:
                    if len(sub_chunk.strip()) > 20:  # Filter very small pieces
                        chunks.append(sub_chunk.strip())
                # Start new chunk with the last piece
                last_piece = sub_chunks[-1].strip()
                if last_piece:
                    current_chunk = [last_piece]
                    current_length = len(last_piece)

            elif current_length + sentence_len <= self.chunk_size:
                # Add sentence to current chunk
                current_chunk.append(sentence)
                current_length += sentence_len
            else:
                # Save current chunk and start new one
                if current_chunk:
                    chunks.append(' '.join(current_chunk))

                # Keep overlap: last few sentences for context
                overlap_sentences = []
                overlap_len = 0
                for s in reversed(current_chunk):
                    if overlap_len + len(s) <= self.chunk_overlap:
                        overlap_sentences.insert(0, s)
                        overlap_len += len(s)
                    else:
                        break

                current_chunk = overlap_sentences + [sentence]
                current_length = overlap_len + sentence_len

        # Add the last chunk
        if current_chunk:
            final_chunk = ' '.join(current_chunk).strip()
            if final_chunk:
                chunks.append(final_chunk)

        # Filter out empty or too-short chunks
        chunks = [chunk for chunk in chunks if len(chunk.strip()) > 50]

        return chunks

    def _split_long_sentence(self, sentence: str) -> List[str]:
        """Split a very long sentence by punctuation or clauses."""
        # Try to split by natural delimiters
        delimiters = ['; ', ', ', ' — ', ' – ', ' and ', ' but ', ' or ', ' however ']

        for delim in delimiters:
            if delim in sentence:
                parts = sentence.split(delim)
                chunks = []
                current = []
                current_len = 0

                for part in parts:
                    part_len = len(part) + len(delim) if current else len(part)
                    if current_len + part_len <= self.chunk_size:
                        current.append(part)
                        current_len += part_len
                    else:
                        if current:
                            chunks.append(delim.join(current))
                        current = [part]
                        current_len = len(part)

                if current:
                    chunks.append(delim.join(current))

                if len(chunks) > 1:
                    return chunks

        # Fallback: simple character split
        words = sentence.split()
        chunks = []
        current = []
        current_len = 0

        for word in words:
            if current_len + len(word) + 1 <= self.chunk_size:
                current.append(word)
                current_len += len(word) + 1
            else:
                if current:
                    chunks.append(' '.join(current))
                current = [word]
                current_len = len(word)

        if current:
            chunks.append(' '.join(current))

        return chunks


# ============================================
# CHUNKING WITH PROGRESS BAR
# ============================================
print("Using sentence-aware sliding window splitter...")
splitter = AdvancedTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    respect_sentences=True  # Set to False for simple sliding window
)

# Process all documents into chunks
chunks: List[Tuple[str, str, int]] = []  # (chunk_text, source_doc, chunk_id)

if documents:
    for doc_hash, doc_text, source in tqdm(documents, desc="Chunking documents"):
        try:
            doc_chunks = splitter.split_text(doc_text)

            for idx, chunk in enumerate(doc_chunks):
                # Basic cleaning: remove excessive whitespace
                chunk = ' '.join(chunk.split())
                if len(chunk) > 50:  # Filter out very small chunks
                    chunks.append((chunk, source, idx))
        except Exception as e:
            print(f"Error chunking {source}: {e}")
            continue
else:
    print("No documents to chunk. Make sure CELL 4 produced some documents.")

# Deduplicate chunks
def remove_duplicate_chunks(chunks: List[Tuple[str, str, int]]) -> List[Tuple[str, str, int]]:
    """Remove exact duplicate chunks."""
    seen_texts = set()
    unique_chunks = []

    for chunk_text, source, idx in chunks:
        # Create a hash of the chunk text (first 500 chars to save memory)
        text_hash = hashlib.sha256(chunk_text.encode('utf-8')).hexdigest()
        if text_hash not in seen_texts:
            seen_texts.add(text_hash)
            unique_chunks.append((chunk_text, source, idx))

    return unique_chunks

# Apply deduplication
original_count = len(chunks)
if chunks:
    chunks = remove_duplicate_chunks(chunks)
    print(f"Removed {original_count - len(chunks)} exact duplicate chunks")
    print(f"Total unique chunks generated: {len(chunks)}")

    # Show statistics
    if chunks:
        chunk_lengths = [len(c[0]) for c in chunks]
        avg_chunk_len = np.mean(chunk_lengths)
        print(f"Average chunk length: {avg_chunk_len:.1f} characters")
        print(f"Chunk size range: {min(chunk_lengths)} - {max(chunk_lengths)} characters")
        print(f"Total characters processed: {sum(chunk_lengths):,}")
else:
    print("No chunks generated. Check if documents have extractable text.")

Using sentence-aware sliding window splitter...


Chunking documents:   0%|          | 0/1 [00:00<?, ?it/s]

Removed 0 exact duplicate chunks
Total unique chunks generated: 25
Average chunk length: 385.5 characters
Chunk size range: 85 - 1023 characters
Total characters processed: 9,637


In [9]:

# GENERATE EMBEDDINGS & BUILD FAISS INDEX

# Load embedding model
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Encode chunks in batches (memory efficient)
batch_size = 64
all_embeddings = []
chunk_metadata = []   # store (chunk_text, source, chunk_id)

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks"):
    batch_chunks = chunks[i:i+batch_size]
    batch_texts = [c[0] for c in batch_chunks]
    batch_emb = embedder.encode(batch_texts, convert_to_numpy=True, show_progress_bar=False)
    all_embeddings.append(batch_emb)
    # store metadata
    for c in batch_chunks:
        chunk_metadata.append({
            "text": c[0],
            "source": c[1],
            "chunk_id": c[2]
        })

embeddings = np.vstack(all_embeddings).astype('float32')
print(f"Embeddings shape: {embeddings.shape}")

# Build FAISS index (inner product = cosine if vectors are normalized)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # Inner product
faiss.normalize_L2(embeddings)   # Normalize for cosine similarity
index.add(embeddings)
print(f"FAISS index size: {index.ntotal} vectors")

# Save index and metadata
faiss.write_index(index, INDEX_PATH)
with open(METADATA_PATH, "wb") as f:
    pickle.dump(chunk_metadata, f)
print(f"Index saved to {INDEX_PATH}, metadata to {METADATA_PATH}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (25, 384)
FAISS index size: 25 vectors
Index saved to /content/faiss_index, metadata to /content/chunks_metadata.pkl


In [11]:
# ================================
# CELL 7: OPTIMIZED FOR COLAB FREE TIER
# ================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch

# Using TinyLlama - specifically designed for constrained environments
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization (required for Colab free)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for speed
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(f"Loading {MODEL_NAME}...")
print("This will take ~2 minutes and use ~2GB GPU RAM")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,  # Reduce memory usage
)

# Create pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.pad_token_id,
)

print(f"✅ Model loaded successfully on {model.device}")
print(f"Memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB" if torch.cuda.is_available() else "CPU mode")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
This will take ~2 minutes and use ~2GB GPU RAM


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'top_p', 'temperature', 'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Model loaded successfully on cuda:0
Memory used: 0.83 GB


In [12]:
# Test the model with a simple prompt
test_prompt = "Hello! How are you?"
try:
    test_response = generator(test_prompt, max_new_tokens=50)
    print("✅ Model test successful!")
    print(f"Response: {test_response[0]['generated_text'][:200]}...")
except Exception as e:
    print(f"⚠️ Model test failed: {e}")
    print("Proceeding anyway - may work with RAG prompts")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Model test successful!
Response: Hello! How are you?...


In [13]:

# RETRIEVAL + GENERATION (RAG PIPELINE)

def retrieve(query: str, k: int = TOP_K) -> List[Dict]:
    """Retrieve top-k relevant chunks given a query."""
    # Encode query
    query_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    # Search
    scores, indices = index.search(query_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        results.append({
            "text": meta["text"],
            "source": meta["source"],
            "score": float(score)
        })
    return results

def build_prompt(query: str, retrieved_chunks: List[Dict]) -> str:
    """Construct prompt with retrieved context."""
    context = "\n\n---\n\n".join([f"[Source: {c['source']}]\n{c['text']}" for c in retrieved_chunks])
    prompt = f"""You are a helpful AI assistant. Answer the user's question using only the provided context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""
    return prompt

def generate_answer(query: str) -> Tuple[str, List[Dict]]:
    """Complete RAG pipeline: retrieve, generate, return answer + sources."""
    retrieved = retrieve(query)
    if not retrieved:
        return "No relevant documents found.", []
    prompt = build_prompt(query, retrieved)
    # Generate response
    response = generator(prompt, max_new_tokens=256, do_sample=True)[0]['generated_text']
    # Extract only the answer part (after "Answer:")
    answer = response.split("Answer:")[-1].strip()
    return answer, retrieved

# Test the pipeline
sample_query = "What is the main contribution of transformer models in NLP?"
answer, sources = generate_answer(sample_query)
print("="*50)
print(f"Query: {sample_query}\n")
print(f"Answer: {answer}\n")
print("Sources:")
for s in sources:
    print(f"- {s['source']} (score: {s['score']:.3f})")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What is the main contribution of transformer models in NLP?

Answer: Transformers are an essential tool for natural language processing (NLP) as they allow for efficient pre-processing, modeling, and prediction of text data. They have made significant progress in recent years, enabling new developments in tasks such as named entity recognition (NER), part-of-speech (POS) tagging, and question answering (QA). They also improve the efficiency of neural machine translation (NMT) by providing a more flexible architecture that allows for more complex language models.

Sources:
- Blockchain security.pdf (score: 0.166)
- Blockchain security.pdf (score: 0.145)
- Blockchain security.pdf (score: 0.122)
- Blockchain security.pdf (score: 0.109)
- Blockchain security.pdf (score: 0.105)


In [14]:

# EVALUATION (Retrieval Metrics on Synthetic QA)

# Simple evaluation: create a few questions from existing chunks and measure hit rate.
# For a real project, use a labeled dataset like Natural Questions.
print("Evaluating retrieval quality...")

test_queries = [
    ("What is a large language model?", "llm"),
    ("Explain backpropagation", "backprop"),
    ("How does attention work in transformers?", "attention")
]

hit_at_5 = 0
for query, expected_keyword in test_queries:
    retrieved = retrieve(query, k=5)
    # Check if any retrieved chunk contains the expected keyword (case-insensitive)
    hits = any(expected_keyword.lower() in r['text'].lower() for r in retrieved)
    if hits:
        hit_at_5 += 1
    print(f"Query: '{query}' -> Hit: {hits}")

print(f"Hit@5 rate: {hit_at_5 / len(test_queries):.2f}")
print("For thorough evaluation, consider using RAGAS or custom answer relevance scores.")

Evaluating retrieval quality...
Query: 'What is a large language model?' -> Hit: False
Query: 'Explain backpropagation' -> Hit: False
Query: 'How does attention work in transformers?' -> Hit: False
Hit@5 rate: 0.00
For thorough evaluation, consider using RAGAS or custom answer relevance scores.


In [15]:

# INTERACTIVE CHAT (Gradio UI)

import gradio as gr

def rag_chat(message, history):
    answer, sources = generate_answer(message)
    source_text = "\n".join([f"📄 {s['source']} (score: {s['score']:.2f})" for s in sources])
    return f"**Answer:** {answer}\n\n**Sources:**\n{source_text}"

iface = gr.ChatInterface(
    fn=rag_chat,
    title="RAG over 1M PDFs (demo with ~20 PDFs)",
    description="Ask questions based on uploaded/downloaded PDFs. The system retrieves relevant chunks and generates an answer."
)
iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4865b55277f8b0b0db.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
